## Data loading and preparation

This notebook builds and evaluates text-based classifiers to predict the desired job position from Russian CVs, using BERT sentence embeddings and classical ML models. In this first section, we import dependencies and load the raw HeadHunter resumes dataset that will later be cleaned, filtered, and transformed into features and labels.

In [1]:
import sys
!"{sys.executable}" -m pip install fairlearn -q

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    equalized_odds_ratio
)
from fairlearn.postprocessing import ThresholdOptimizer

RANDOM_STATE = 42

In [3]:
# for graphs
import sys
!"{sys.executable}" -m pip install matplotlib -q
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

In [4]:
import os
import pandas as pd

BASE_DIR = ".."
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

df_train = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
df_val   = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
df_test  = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print("Train:", df_train.shape)
print("Val:", df_val.shape)
print("Test:", df_test.shape)

Train: (16530, 5)
Val: (5510, 5)
Test: (5510, 5)


In [5]:
mapping = pd.read_csv("../data/processed/label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))

df_train["supercategory"] = df_train["label"].map(label_to_supercat)
df_val["supercategory"] = df_val["label"].map(label_to_supercat)
df_test["supercategory"] = df_test["label"].map(label_to_supercat)

print(df_train["supercategory"].value_counts())

# sanity check: mapping coverage
for name, d in [("train", df_train), ("val", df_val), ("test", df_test)]:
    n_nan = d["supercategory"].isna().sum()
    print(name, "NaN supercategory:", n_nan)

# if there are any missing labels, show them
missing = set(df_train.loc[df_train["supercategory"].isna(), "label"].unique())
if missing:
    print("Missing labels (train):", sorted(list(missing))[:50])

supercategory
sysadmin_devops_network     3308
generic_it_ops              2913
it_governance_leadership    2129
project_product             2006
technical_specialized       1982
backend_general_dev         1546
sales_account               1029
tech_support_helpdesk       1016
web_frontend                 601
Name: count, dtype: int64
train NaN supercategory: 0
val NaN supercategory: 0
test NaN supercategory: 0


In [8]:
USE_SUPERCATEGORIES = True

if USE_SUPERCATEGORIES:
    target_column = "supercategory"
else:
    target_column = "label"

X_train = df_train["resume_text"].values
y_train = df_train[target_column].values

X_val = df_val["resume_text"].values
y_val = df_val[target_column].values

X_test = df_test["resume_text"].values
y_test = df_test[target_column].values

len(set(y_train))

9

## Sentence-transformer embeddings and baseline classifier

In this section we download a pre-trained sentence-transformer model (`paraphrase-MiniLM-L3-v2`) from Hugging Face and use it to encode each resume into a dense vector representation. On top of these fixed embeddings, we first train a simple logistic regression classifier and evaluate its accuracy and macro F1 on the test set as an initial baseline across many profession classes.

In [9]:
import sys
!"{sys.executable}" -m pip install sentence-transformers -q

In [10]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="sentence-transformers/paraphrase-MiniLM-L3-v2",
    local_dir="./local_bert",
    local_dir_use_symlinks=False
)

/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_di

'/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/notebooks/local_bert'

In [11]:
from sentence_transformers import SentenceTransformer
bert = SentenceTransformer("./local_bert")

In [12]:
X_train_emb = bert.encode(X_train, batch_size=32, show_progress_bar=True)
X_val_emb   = bert.encode(X_val, batch_size=32, show_progress_bar=True)
X_test_emb  = bert.encode(X_test, batch_size=32, show_progress_bar=True)

print("Train:", X_train_emb.shape)
print("Val:", X_val_emb.shape)
print("Test:", X_test_emb.shape)

Batches: 100%|██████████| 173/173 [00:24<00:00,  7.15it/s]


Train: (16530, 384)
Val: (5510, 384)
Test: (5510, 384)


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# classifier
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train_emb, y_train)

# predictions
y_test_pred = clf.predict(X_test_emb)

# metrics
acc = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred, average="macro")

print("Accuracy:", acc)
print("F1 (macro):", f1)

Accuracy: 0.4903811252268603
F1 (macro): 0.4919256330024623


## Focusing on frequent professions and stronger classifier

Finally, we restrict the problem to a subset of more frequent professions to reduce label noise and class sparsity, rebuild the dataset, and re-encode resumes with the same sentence-transformer. We then train a stronger linear SVM classifier on this reduced label space and compare its performance (accuracy and macro F1) to the previous logistic regression baseline to understand how class filtering and model choice affect results.

In [41]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

svm = LinearSVC()
svm.fit(X_train_emb, y_train)

y_test_pred = svm.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1 (macro):", f1_score(y_test, y_test_pred, average="macro"))

Accuracy: 0.3952813067150635
F1 (macro): 0.31667626883316763


In [42]:
test_df = df_test.copy()
test_df["y_true"] = y_test
test_df["y_pred"] = y_test_pred

In [43]:
from sklearn.metrics import accuracy_score

def group_accuracy(df, group_col):
    results = []
    for group in sorted(df[group_col].dropna().unique()):
        group_df = df[df[group_col] == group]
        acc = accuracy_score(group_df["y_true"], group_df["y_pred"])
        results.append((group, len(group_df), round(acc, 3)))
    return results

In [44]:
# conclusion on every sensitive attribut

print("Accuracy by city:")
for row in group_accuracy(test_df, "city_group"):
    print(row)

print("\nAccuracy by gender:")
for row in group_accuracy(test_df, "gender"):
    print(row)

print("\nAccuracy by age group:")
for row in group_accuracy(test_df, "age_group"):
    print(row)

Accuracy by city:
('Other', 1312, 0.384)
('Алматы', 67, 0.701)
('Барнаул', 21, 0.476)
('Владивосток', 21, 0.333)
('Волгоград', 25, 0.4)
('Воронеж', 72, 0.444)
('Екатеринбург', 89, 0.416)
('Ижевск', 19, 0.421)
('Иркутск', 33, 0.364)
('Казань', 103, 0.34)
('Калининград', 23, 0.609)
('Калуга', 18, 0.278)
('Кемерово', 22, 0.409)
('Краснодар', 158, 0.367)
('Красноярск', 49, 0.469)
('Липецк', 25, 0.32)
('Минск', 31, 0.806)
('Москва', 1842, 0.414)
('Набережные Челны', 21, 0.333)
('Нижний Новгород', 83, 0.205)
('Новосибирск', 126, 0.341)
('Омск', 45, 0.222)
('Оренбург', 31, 0.387)
('Пенза', 20, 0.35)
('Пермь', 56, 0.429)
('Ростов-на-Дону', 76, 0.329)
('Самара', 107, 0.458)
('Санкт-Петербург', 568, 0.382)
('Саратов', 33, 0.424)
('Симферополь', 18, 0.611)
('Сочи', 22, 0.318)
('Ставрополь', 27, 0.407)
('Тверь', 22, 0.273)
('Тольятти', 30, 0.333)
('Томск', 41, 0.415)
('Тюмень', 47, 0.277)
('Ульяновск', 20, 0.2)
('Уфа', 84, 0.357)
('Хабаровск', 26, 0.538)
('Челябинск', 29, 0.31)
('Ярославль', 48, 0

In [45]:
print("Overall test accuracy:",
      accuracy_score(y_test, y_test_pred))

Overall test accuracy: 0.3952813067150635
